In [1]:
import sys
sys.path.append('..')

from src.preprocess import preprocess
from src.features import select_features, engineer_features
from src.dimensionality import apply_pca, train_autoencoder
from src.models.random_forest import train_random_forest, evaluate as rf_evaluate, save_model as rf_save
from src.models.svm import train_svm, evaluate as svm_evaluate, save_model as svm_save
from src.models.lstm import train_lstm, evaluate as lstm_evaluate, save_model as lstm_save
from src.ensemble import run_ensemble
from src.evaluate import compute_metrics, plot_confusion_matrix, plot_training_history, compare_models

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print('All imports OK')

All imports OK


In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Load only 2 files to keep memory manageable
files = [
    '../data/raw/cicids2017/Monday-WorkingHours.pcap_ISCX.csv',
    '../data/raw/cicids2017/Tuesday-WorkingHours.pcap_ISCX.csv',
]

dfs = []
for f in files:
    df = pd.read_csv(f, encoding='utf-8', low_memory=False)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    print(f"Loaded {os.path.basename(f)}: {df.shape}")
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

# Clean
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
print(f"\nCleaned shape: {df.shape}")

# Sample down to 200k rows to keep memory low
if len(df) > 200000:
    df = df.groupby('label', group_keys=False).apply(
        lambda x: x.sample(min(len(x), 20000), random_state=42)
    )
    print(f"Sampled shape: {df.shape}")

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['label'])
label_names = list(le.classes_)
num_classes = len(label_names)
print(f"\nClasses: {label_names}")

# Split
X = df.drop(columns=['label']).select_dtypes(include=[np.number])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"\nX_train: {X_train.shape}, X_test: {X_test.shape}")

Loaded Monday-WorkingHours.pcap_ISCX.csv: (529918, 79)
Loaded Tuesday-WorkingHours.pcap_ISCX.csv: (445909, 79)

Cleaned shape: (924226, 79)
Sampled shape: (29150, 79)

Classes: ['BENIGN', 'FTP-Patator', 'SSH-Patator']

X_train: (23320, 78), X_test: (5830, 78)


In [4]:
from src.dimensionality import apply_pca

X_train_pca, X_test_pca, pca = apply_pca(X_train, X_test, variance_threshold=0.95)
print(f'After PCA: {X_train_pca.shape}')

PCA: 78 features → 23 components (95% variance retained)
After PCA: (23320, 23)


In [5]:
rf_model = train_random_forest(X_train_pca, y_train)
rf_pred, rf_cm = rf_evaluate(rf_model, X_test_pca, y_test, label_names)
rf_metrics = compute_metrics(y_test, rf_pred, model_name='Random Forest')
plot_confusion_matrix(y_test, rf_pred, label_names, model_name='Random Forest')
rf_save(rf_model)

Training Random Forest...
Random Forest training complete.

--- Random Forest Results ---
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00      4000
 FTP-Patator       1.00      1.00      1.00      1186
 SSH-Patator       1.00      1.00      1.00       644

    accuracy                           1.00      5830
   macro avg       1.00      1.00      1.00      5830
weighted avg       1.00      1.00      1.00      5830

Confusion Matrix:
 [[4000    0    0]
 [   0 1183    3]
 [   1    0  643]]

[Random Forest] Accuracy: 0.9993 | F1: 0.9993 | Precision: 0.9993 | Recall: 0.9993 | FPR: 0.0004
Saved confusion matrix to results\cm_random_forest.png
Model saved to results/rf_model.pkl


In [6]:
svm_model = train_svm(X_train_pca, y_train)
svm_pred, svm_cm = svm_evaluate(svm_model, X_test_pca, y_test, label_names)
svm_metrics = compute_metrics(y_test, svm_pred, model_name='SVM')
plot_confusion_matrix(y_test, svm_pred, label_names, model_name='SVM')
svm_save(svm_model)

Training SVM...
SVM training complete.

--- SVM Results ---
              precision    recall  f1-score   support

      BENIGN       1.00      0.98      0.99      4000
 FTP-Patator       0.97      0.99      0.98      1186
 SSH-Patator       0.90      0.99      0.95       644

    accuracy                           0.98      5830
   macro avg       0.96      0.99      0.97      5830
weighted avg       0.98      0.98      0.98      5830

Confusion Matrix:
 [[3905   31   64]
 [   3 1180    3]
 [   3    3  638]]

[SVM] Accuracy: 0.9816 | F1: 0.9819 | Precision: 0.9828 | Recall: 0.9816 | FPR: 0.0078
Saved confusion matrix to results\cm_svm.png
Model saved to results/svm_model.pkl


In [7]:
lstm_model, history = train_lstm(X_train_pca, y_train, X_test_pca, y_test, num_classes)
lstm_pred, lstm_cm = lstm_evaluate(lstm_model, X_test_pca, y_test, label_names)
lstm_metrics = compute_metrics(y_test, lstm_pred, model_name='LSTM')
plot_confusion_matrix(y_test, lstm_pred, label_names, model_name='LSTM')
plot_training_history(history)
lstm_save(lstm_model)

Training LSTM...
Model: "lstm_ids"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 1, 128)            77824     
                                                                 
 dropout (Dropout)           (None, 1, 128)            0         
                                                                 
 lstm_1 (LSTM)               (None, 64)                49408     
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense (Dense)               (None, 64)                4160      
                                                                 
 dense_1 (Dense)             (None, 3)                 195       
                                                                 
Total params: 131587 (514.01 KB)
Trainabl

INFO:tensorflow:Assets written to: results/lstm_model\assets


Model saved to results/lstm_model


In [8]:
hard_pred, weighted_pred = run_ensemble(
    rf_model, svm_model, lstm_model,
    X_test_pca, y_test,
    weights=(0.3, 0.2, 0.5),
    label_names=label_names
)
hard_metrics     = compute_metrics(y_test, hard_pred,     model_name='Hard Vote')
weighted_metrics = compute_metrics(y_test, weighted_pred, model_name='Weighted Vote')

183/183 [==============================] - 1s 6ms/step

Running hard voting...

--- HARD VOTE Results ---
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00      4000
 FTP-Patator       1.00      0.99      1.00      1186
 SSH-Patator       0.99      0.99      0.99       644

    accuracy                           1.00      5830
   macro avg       1.00      1.00      1.00      5830
weighted avg       1.00      1.00      1.00      5830

Confusion Matrix:
 [[3996    3    1]
 [   3 1180    3]
 [   2    2  640]]

Running weighted voting...

--- WEIGHTED VOTE Results ---
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00      4000
 FTP-Patator       0.99      0.99      0.99      1186
 SSH-Patator       0.99      0.99      0.99       644

    accuracy                           1.00      5830
   macro avg       1.00      1.00      1.00      5830
weighted avg       1.00      1.00      1.00      5830


In [9]:
# Compare all models
all_metrics = [rf_metrics, svm_metrics, lstm_metrics, hard_metrics, weighted_metrics]
compare_models(all_metrics)

import pandas as pd
results_df = pd.DataFrame(all_metrics).set_index('model')
print(results_df)

# Save scaler, PCA and label encoder for the dashboard
import joblib, os
os.makedirs('results', exist_ok=True)
joblib.dump(scaler, '../results/scaler.pkl')
joblib.dump(pca,    '../results/pca.pkl')
joblib.dump(le,     '../results/label_encoder.pkl')
print('Scaler, PCA and label encoder saved.')

Saved model comparison chart to results\model_comparison.png
               accuracy  f1_score  precision  recall     fpr
model                                                       
Random Forest    0.9993    0.9993     0.9993  0.9993  0.0004
SVM              0.9816    0.9819     0.9828  0.9816  0.0078
LSTM             0.9969    0.9969     0.9969  0.9969  0.0023
Hard Vote        0.9976    0.9976     0.9976  0.9976  0.0015
Weighted Vote    0.9976    0.9976     0.9976  0.9976  0.0014
Scaler, PCA and label encoder saved.


In [10]:
# Load all 8 files for more attack variety
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

files = [f for f in os.listdir('../data/raw/cicids2017/') if f.endswith('.csv')]
print(f"Found {len(files)} files:")
for f in files:
    print(f" - {f}")

Found 8 files:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv


In [11]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

data_dir = '../data/raw/cicids2017/'
files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]

dfs = []
for f in files:
    df = pd.read_csv(os.path.join(data_dir, f), encoding='utf-8', low_memory=False)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)
    print(f"{f}: {df.shape} | Classes: {df['label'].unique()}")
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nFull dataset: {df_all.shape}")
print(f"\nClass distribution:\n{df_all['label'].value_counts()}")

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: (223082, 79) | Classes: ['BENIGN' 'DDoS']
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: (213777, 79) | Classes: ['BENIGN' 'PortScan']
Friday-WorkingHours-Morning.pcap_ISCX.csv: (184044, 79) | Classes: ['BENIGN' 'Bot']
Monday-WorkingHours.pcap_ISCX.csv: (502650, 79) | Classes: ['BENIGN']
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: (252790, 79) | Classes: ['BENIGN' 'Infiltration']
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: (164179, 79) | Classes: ['BENIGN' 'Web Attack � Brute Force' 'Web Attack � XSS'
 'Web Attack � Sql Injection']
Tuesday-WorkingHours.pcap_ISCX.csv: (421626, 79) | Classes: ['BENIGN' 'FTP-Patator' 'SSH-Patator']
Wednesday-workingHours.pcap_ISCX.csv: (610492, 79) | Classes: ['BENIGN' 'DoS slowloris' 'DoS Slowhttptest' 'DoS Hulk' 'DoS GoldenEye'
 'Heartbleed']

Full dataset: (2572640, 79)

Class distribution:
BENIGN                        2146899
DoS Hulk                       172846
DD

In [12]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

folder = '../data/raw/cicids2017/'
files  = [f for f in os.listdir(folder) if f.endswith('.csv')]

dfs = []
for f in files:
    df = pd.read_csv(os.path.join(folder, f), encoding='utf-8', low_memory=False)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)
    print(f"{f}: {df.shape} | Classes: {df['label'].unique()}")
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nFull dataset: {df_all.shape}")
print("\nClass distribution:")
print(df_all['label'].value_counts())

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: (223082, 79) | Classes: ['BENIGN' 'DDoS']
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: (213777, 79) | Classes: ['BENIGN' 'PortScan']
Friday-WorkingHours-Morning.pcap_ISCX.csv: (184044, 79) | Classes: ['BENIGN' 'Bot']
Monday-WorkingHours.pcap_ISCX.csv: (502650, 79) | Classes: ['BENIGN']
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: (252790, 79) | Classes: ['BENIGN' 'Infiltration']
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: (164179, 79) | Classes: ['BENIGN' 'Web Attack � Brute Force' 'Web Attack � XSS'
 'Web Attack � Sql Injection']
Tuesday-WorkingHours.pcap_ISCX.csv: (421626, 79) | Classes: ['BENIGN' 'FTP-Patator' 'SSH-Patator']
Wednesday-workingHours.pcap_ISCX.csv: (610492, 79) | Classes: ['BENIGN' 'DoS slowloris' 'DoS Slowhttptest' 'DoS Hulk' 'DoS GoldenEye'
 'Heartbleed']

Full dataset: (2572640, 79)

Class distribution:
BENIGN                        2146899
DoS Hulk                       172846
DD

In [13]:
# Smart sampling — cap each class to avoid RAM issues
df_sampled = df_all.groupby('label', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 5000), random_state=42)
)
print(f"Sampled shape: {df_sampled.shape}")
print("\nSampled class distribution:")
print(df_sampled['label'].value_counts())

# Drop rare classes with fewer than 20 samples (Heartbleed, SQL Injection)
class_counts = df_sampled['label'].value_counts()
valid_classes = class_counts[class_counts >= 20].index
df_sampled = df_sampled[df_sampled['label'].isin(valid_classes)]
print(f"\nAfter dropping rare classes: {df_sampled.shape}")

# Re-encode labels
le2 = LabelEncoder()
df_sampled['label'] = le2.fit_transform(df_sampled['label'])
label_names2 = list(le2.classes_)
num_classes2  = len(label_names2)
print(f"\nClasses ({num_classes2}): {label_names2}")

# Split
X2 = df_sampled.drop(columns=['label']).select_dtypes(include=[np.number])
y2 = df_sampled['label']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

# Scale
scaler2 = StandardScaler()
X2_train = scaler2.fit_transform(X2_train)
X2_test  = scaler2.transform(X2_test)

print(f"\nX_train: {X2_train.shape}, X_test: {X2_test.shape}")

Sampled shape: (47357, 79)

Sampled class distribution:
BENIGN                        5000
DDoS                          5000
DoS GoldenEye                 5000
DoS Hulk                      5000
DoS Slowhttptest              5000
DoS slowloris                 5000
FTP-Patator                   5000
PortScan                      5000
SSH-Patator                   3219
Bot                           1948
Web Attack � Brute Force      1470
Web Attack � XSS               652
Infiltration                    36
Web Attack � Sql Injection      21
Heartbleed                      11
Name: label, dtype: int64

After dropping rare classes: (47346, 79)

Classes (14): ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'Infiltration', 'PortScan', 'SSH-Patator', 'Web Attack � Brute Force', 'Web Attack � Sql Injection', 'Web Attack � XSS']

X_train: (37876, 78), X_test: (9470, 78)


In [14]:
# PCA
from src.dimensionality import apply_pca
X2_train_pca, X2_test_pca, pca2 = apply_pca(X2_train, X2_test, variance_threshold=0.95)
print(f'After PCA: {X2_train_pca.shape}')

PCA: 78 features → 21 components (95% variance retained)
After PCA: (37876, 21)


In [15]:
# Random Forest
rf2 = train_random_forest(X2_train_pca, y2_train)
rf2_pred, _ = rf_evaluate(rf2, X2_test_pca, y2_test, label_names2)
rf2_metrics = compute_metrics(y2_test, rf2_pred, model_name='RF v2')
plot_confusion_matrix(y2_test, rf2_pred, label_names2, model_name='RF v2')
rf_save(rf2, path='results/rf2_model.pkl')

Training Random Forest...
Random Forest training complete.

--- Random Forest Results ---
                            precision    recall  f1-score   support

                    BENIGN       0.97      0.97      0.97      1000
                       Bot       0.98      0.99      0.98       390
                      DDoS       1.00      1.00      1.00      1000
             DoS GoldenEye       0.99      0.99      0.99      1000
                  DoS Hulk       1.00      0.99      0.99      1000
          DoS Slowhttptest       0.99      0.99      0.99      1000
             DoS slowloris       1.00      0.99      0.99      1000
               FTP-Patator       1.00      1.00      1.00      1000
              Infiltration       1.00      0.86      0.92         7
                  PortScan       1.00      1.00      1.00      1000
               SSH-Patator       0.98      0.99      0.98       644
  Web Attack � Brute Force       0.72      0.78      0.75       294
Web Attack � Sql Injectio

In [16]:
# LSTM
lstm2, history2 = train_lstm(X2_train_pca, y2_train, X2_test_pca, y2_test, num_classes2)
lstm2_pred, _ = lstm_evaluate(lstm2, X2_test_pca, y2_test, label_names2)
lstm2_metrics = compute_metrics(y2_test, lstm2_pred, model_name='LSTM v2')
plot_confusion_matrix(y2_test, lstm2_pred, label_names2, model_name='LSTM v2')
plot_training_history(history2, model_name='LSTM v2')
lstm_save(lstm2, path='results/lstm2_model')

Training LSTM...
Model: "lstm_ids"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_2 (LSTM)               (None, 1, 128)            76800     
                                                                 
 dropout_2 (Dropout)         (None, 1, 128)            0         
                                                                 
 lstm_3 (LSTM)               (None, 64)                49408     
                                                                 
 dropout_3 (Dropout)         (None, 64)                0         
                                                                 
 dense_2 (Dense)             (None, 64)                4160      
                                                                 
 dense_3 (Dense)             (None, 14)                910       
                                                                 
Total params: 131278 (512.80 KB)
Trainabl

INFO:tensorflow:Assets written to: results/lstm2_model\assets


Model saved to results/lstm2_model


In [21]:
# Reload what we have
import joblib
from tensorflow import keras

rf2   = joblib.load('results/rf2_model.pkl')
lstm2 = keras.models.load_model('results/lstm2_model')
print('RF and LSTM reloaded.')


RF and LSTM reloaded.


In [22]:
print(X2_train_pca.shape)

(37876, 21)


In [23]:
# Retrain SVM v2 (kernel was lost)
from src.models.svm import train_svm, evaluate as svm_evaluate, save_model as svm_save

svm2 = train_svm(X2_train_pca, y2_train)
svm2_pred, _ = svm_evaluate(svm2, X2_test_pca, y2_test, label_names2)
svm2_metrics = compute_metrics(y2_test, svm2_pred, model_name='SVM v2')
svm_save(svm2, path='results/svm2_model.pkl')

Training SVM...
SVM training complete.

--- SVM Results ---
                            precision    recall  f1-score   support

                    BENIGN       0.92      0.84      0.88      1000
                       Bot       0.92      0.98      0.95       390
                      DDoS       0.97      1.00      0.99      1000
             DoS GoldenEye       0.98      0.95      0.96      1000
                  DoS Hulk       0.98      0.95      0.97      1000
          DoS Slowhttptest       0.98      0.97      0.98      1000
             DoS slowloris       0.97      0.95      0.96      1000
               FTP-Patator       0.97      1.00      0.99      1000
              Infiltration       0.38      0.71      0.50         7
                  PortScan       0.98      0.99      0.98      1000
               SSH-Patator       0.99      0.91      0.95       644
  Web Attack � Brute Force       0.75      0.05      0.10       294
Web Attack � Sql Injection       0.02      1.00      0.

In [24]:
hard2_pred, weighted2_pred = run_ensemble(
    rf2, svm2, lstm2,
    X2_test_pca, y2_test,
    weights=(0.3, 0.2, 0.5),
    label_names=label_names2
)
hard2_metrics     = compute_metrics(y2_test, hard2_pred,     model_name='Hard Vote v2')
weighted2_metrics = compute_metrics(y2_test, weighted2_pred, model_name='Weighted Vote v2')

all_metrics2 = [rf2_metrics, svm2_metrics, lstm2_metrics, hard2_metrics, weighted2_metrics]
compare_models(all_metrics2)

import pandas as pd
pd.DataFrame(all_metrics2).set_index('model')

296/296 [==============================] - 5s 7ms/step

Running hard voting...

--- HARD VOTE Results ---
                            precision    recall  f1-score   support

                    BENIGN       0.98      0.91      0.94      1000
                       Bot       0.94      1.00      0.97       390
                      DDoS       1.00      1.00      1.00      1000
             DoS GoldenEye       0.98      1.00      0.99      1000
                  DoS Hulk       0.98      0.99      0.98      1000
          DoS Slowhttptest       0.99      0.99      0.99      1000
             DoS slowloris       0.99      0.98      0.98      1000
               FTP-Patator       0.99      0.99      0.99      1000
              Infiltration       1.00      0.71      0.83         7
                  PortScan       0.98      0.99      0.99      1000
               SSH-Patator       0.98      0.98      0.98       644
  Web Attack � Brute Force       0.67      0.98      0.79       294
Web Attac

,accuracy,f1_score,precision,recall,fpr
model,,,,,
RF v2,0.9750,0.9743,0.9738,0.9750,0.0019
SVM v2,0.9251,0.9268,0.9533,0.9251,0.0056
LSTM v2,0.9660,0.9608,0.9702,0.9660,0.0026
Hard Vote v2,0.9682,0.9629,0.9728,0.9682,0.0024
Weighted Vote v2,0.9693,0.9653,0.9672,0.9693,0.0023


In [25]:
import joblib
joblib.dump(scaler2, 'results/scaler2.pkl')
joblib.dump(pca2,    'results/pca2.pkl')
joblib.dump(le2,     'results/label_encoder2.pkl')
print('V2 artifacts saved.')

V2 artifacts saved.


In [2]:
import joblib
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

folder = '../data/raw/cicids2017/'
files  = [f for f in os.listdir(folder) if f.endswith('.csv')]

dfs = []
for f in files:
    df = pd.read_csv(os.path.join(folder, f), encoding='utf-8', low_memory=False)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

df_sampled = df_all.groupby('label', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 5000), random_state=42)
)
class_counts = df_sampled['label'].value_counts()
valid_classes = class_counts[class_counts >= 20].index
df_sampled = df_sampled[df_sampled['label'].isin(valid_classes)]

le2 = LabelEncoder()
df_sampled['label'] = le2.fit_transform(df_sampled['label'])

X2 = df_sampled.drop(columns=['label']).select_dtypes(include=[np.number])
y2 = df_sampled['label']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2)

scaler2 = StandardScaler()
X2_train_s = scaler2.fit_transform(X2_train)
X2_test_s  = scaler2.transform(X2_test)

from sklearn.decomposition import PCA
pca2 = PCA(n_components=0.95, random_state=42)
X2_train_pca = pca2.fit_transform(X2_train_s)
X2_test_pca  = pca2.transform(X2_test_s)

from sklearn.svm import SVC
svm2 = SVC(kernel='rbf', C=1.0, random_state=42, class_weight='balanced', probability=True)
svm2.fit(X2_train_pca, y2_train)

joblib.dump(svm2, 'results/svm2_model.pkl')
print('SVM saved:', os.path.getsize('results/svm2_model.pkl'), 'bytes')

SVM saved: 3041619 bytes


In [1]:
import joblib
import numpy as np
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

folder = '../data/raw/cicids2017/'
files  = [f for f in os.listdir(folder) if f.endswith('.csv')]

dfs = []
for f in files:
    df = pd.read_csv(os.path.join(folder, f), encoding='utf-8', low_memory=False)
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_sampled = df_all.groupby('label', group_keys=False).apply(
    lambda x: x.sample(min(len(x), 5000), random_state=42))
class_counts = df_sampled['label'].value_counts()
df_sampled = df_sampled[df_sampled['label'].isin(class_counts[class_counts >= 20].index)]

le2 = LabelEncoder()
df_sampled['label'] = le2.fit_transform(df_sampled['label'])
X2 = df_sampled.drop(columns=['label']).select_dtypes(include=[np.number])
y2 = df_sampled['label']

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42, stratify=y2)
scaler2 = StandardScaler()
X2_train_s = scaler2.fit_transform(X2_train)
pca2 = PCA(n_components=0.95, random_state=42)
X2_train_pca = pca2.fit_transform(X2_train_s)

rf2 = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf2.fit(X2_train_pca, y2_train)

joblib.dump(rf2,     'results/rf2_model.pkl')
joblib.dump(scaler2, 'results/scaler2.pkl')
joblib.dump(pca2,    'results/pca2.pkl')
joblib.dump(le2,     'results/label_encoder2.pkl')
print('Done:', joblib.load('results/rf2_model.pkl'))

Done: RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)
